In [0]:
from pyspark.sql import functions as F

In [0]:
df_bronze = spark.table("kbeauty_bronze_products")

display(df_bronze)

brand,category,in_stock,key_ingredients,price_brl,product_id,product_name,rating
Medicube,Beauty_Device,true,"List(PDRN, Peptides)","R$ 1.899,90",KB001,AGE-R Booster Pro X2,4.8
MEDICUBE,skincare,TRUE,List(PDRN),"R$ 149,90",KB002,AGE-R PDRN Booster Gel,4.7
COSRX,SKINCARE,false,List(Hydrocolloid),39.90,KB003,Acne Master Patch,4.6
Sulwhasoo,Skincare,true,List(Ginseng),"R$ 89,90",KB004,Concentrated Ginseng Mask,null
IOPE,serum,true,List(Vitamin C),null,KB005,Vitamin C Expert 25 Ampoule,4.9
IOPE,serum,true,List(Vitamin C),null,KB005,Vitamin C Expert 25 Ampoule,4.9


In [0]:
df_silver = df_bronze.dropDuplicates(["product_id"])

display(df_silver)

brand,category,in_stock,key_ingredients,price_brl,product_id,product_name,rating
Medicube,Beauty_Device,true,"List(PDRN, Peptides)","R$ 1.899,90",KB001,AGE-R Booster Pro X2,4.8
MEDICUBE,skincare,TRUE,List(PDRN),"R$ 149,90",KB002,AGE-R PDRN Booster Gel,4.7
COSRX,SKINCARE,false,List(Hydrocolloid),39.90,KB003,Acne Master Patch,4.6
Sulwhasoo,Skincare,true,List(Ginseng),"R$ 89,90",KB004,Concentrated Ginseng Mask,null
IOPE,serum,true,List(Vitamin C),null,KB005,Vitamin C Expert 25 Ampoule,4.9


In [0]:
print("Bronze rows:", df_bronze.count())
print("Silver rows after deduplication:", df_silver.count())

Bronze rows: 6
Silver rows after deduplication: 5


In [0]:
df_silver = (
    df_silver
    .withColumn(
        "brand",
        F.initcap(F.lower(F.col("brand")))
    )
    .withColumn(
        "category",
        F.when(
            F.lower(F.col("category")).isin("skincare", "serum"),
            "Skincare"
        )
        .when(
            F.lower(F.col("category")) == "beauty_device",
            "Beauty Device"
        )
        .otherwise(
            F.initcap(F.lower(F.col("category")))
        )
    )
)

display(df_silver)

brand,category,in_stock,key_ingredients,price_brl,product_id,product_name,rating
Medicube,Beauty Device,true,"List(PDRN, Peptides)","R$ 1.899,90",KB001,AGE-R Booster Pro X2,4.8
Medicube,Skincare,TRUE,List(PDRN),"R$ 149,90",KB002,AGE-R PDRN Booster Gel,4.7
Cosrx,Skincare,false,List(Hydrocolloid),39.90,KB003,Acne Master Patch,4.6
Sulwhasoo,Skincare,true,List(Ginseng),"R$ 89,90",KB004,Concentrated Ginseng Mask,null
Iope,Skincare,true,List(Vitamin C),null,KB005,Vitamin C Expert 25 Ampoule,4.9


In [0]:
df_silver.select(
    "brand",
    "category"
).distinct().show()

+---------+-------------+
|    brand|     category|
+---------+-------------+
| Medicube|Beauty Device|
| Medicube|     Skincare|
|    Cosrx|     Skincare|
|Sulwhasoo|     Skincare|
|     Iope|     Skincare|
+---------+-------------+



In [0]:
df_silver = (
    df_silver
    .withColumn(
        "price_brl",
        F.regexp_replace(
            F.regexp_replace(
                F.col("price_brl"),
                "R\\$\\s*",
                ""
            ),
            "\\.",
            ""
        )
    )
    .withColumn(
        "price_brl",
        F.regexp_replace(
            F.col("price_brl"),
            ",",
            "."
        )
    )
    .withColumn(
        "price_brl",
        F.col("price_brl").cast("double")
    )
    .withColumn(
        "rating",
        F.col("rating").cast("double")
    )
    .withColumn(
        "in_stock",
        F.lower(F.col("in_stock")) == "true"
    )
)

display(df_silver)

brand,category,in_stock,key_ingredients,price_brl,product_id,product_name,rating
Medicube,Beauty Device,true,"List(PDRN, Peptides)",1899.9,KB001,AGE-R Booster Pro X2,4.8
Medicube,Skincare,true,List(PDRN),149.9,KB002,AGE-R PDRN Booster Gel,4.7
Cosrx,Skincare,false,List(Hydrocolloid),3990.0,KB003,Acne Master Patch,4.6
Sulwhasoo,Skincare,true,List(Ginseng),89.9,KB004,Concentrated Ginseng Mask,null
Iope,Skincare,true,List(Vitamin C),null,KB005,Vitamin C Expert 25 Ampoule,4.9


In [0]:
df_silver.printSchema()

root
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- in_stock: boolean (nullable = true)
 |-- key_ingredients: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- price_brl: double (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- rating: double (nullable = true)



In [0]:
df_silver = (
    df_silver
    .withColumn(
        "price_brl",
        F.when(
            F.col("price_brl").contains(","),
            F.regexp_replace(
                F.regexp_replace(
                    F.regexp_replace(
                        F.col("price_brl"),
                        "R\\$\\s*",
                        ""
                    ),
                    "\\.",
                    ""
                ),
                ",",
                "."
            )
        )
        .otherwise(
            F.regexp_replace(
                F.col("price_brl"),
                "R\\$\\s*",
                ""
            )
        )
    )
    .withColumn(
        "price_brl",
        F.col("price_brl").cast("double")
    )
    .withColumn(
        "rating",
        F.col("rating").cast("double")
    )
    .withColumn(
        "in_stock",
        F.lower(F.col("in_stock").cast("string")) == "true"
    )
)

In [0]:
display(
    df_silver.select(
        "product_id",
        "product_name",
        "price_brl",
        "rating",
        "in_stock"
    )
)

product_id,product_name,price_brl,rating,in_stock
KB001,AGE-R Booster Pro X2,1899.9,4.8,true
KB002,AGE-R PDRN Booster Gel,149.9,4.7,true
KB003,Acne Master Patch,3990.0,4.6,false
KB004,Concentrated Ginseng Mask,89.9,null,true
KB005,Vitamin C Expert 25 Ampoule,null,4.9,true


In [0]:
df_silver = (
    df_bronze
    .dropDuplicates(["product_id"])

    # Standardize brand
    .withColumn(
        "brand",
        F.initcap(F.lower(F.col("brand")))
    )

    # Standardize category
    .withColumn(
        "category",
        F.when(
            F.lower(F.col("category")).isin("skincare", "serum"),
            "Skincare"
        )
        .when(
            F.lower(F.col("category")) == "beauty_device",
            "Beauty Device"
        )
        .otherwise(
            F.initcap(F.lower(F.col("category")))
        )
    )

    # Clean BRL price
    .withColumn(
        "price_brl",
        F.when(
            F.col("price_brl").contains(","),
            F.regexp_replace(
                F.regexp_replace(
                    F.regexp_replace(
                        F.col("price_brl"),
                        "R\\$\\s*",
                        ""
                    ),
                    "\\.",
                    ""
                ),
                ",",
                "."
            )
        )
        .otherwise(
            F.regexp_replace(
                F.col("price_brl"),
                "R\\$\\s*",
                ""
            )
        )
    )

    # Correct data types
    .withColumn(
        "price_brl",
        F.col("price_brl").cast("double")
    )
    .withColumn(
        "rating",
        F.col("rating").cast("double")
    )
    .withColumn(
        "in_stock",
        F.lower(F.col("in_stock")) == "true"
    )
)

display(
    df_silver.select(
        "product_id",
        "product_name",
        "price_brl",
        "rating",
        "in_stock"
    )
)

product_id,product_name,price_brl,rating,in_stock
KB001,AGE-R Booster Pro X2,1899.9,4.8,true
KB002,AGE-R PDRN Booster Gel,149.9,4.7,true
KB003,Acne Master Patch,39.9,4.6,false
KB004,Concentrated Ginseng Mask,89.9,null,true
KB005,Vitamin C Expert 25 Ampoule,null,4.9,true


In [0]:
df_silver = (
    df_silver
    .withColumn(
        "has_missing_price",
        F.col("price_brl").isNull()
    )
    .withColumn(
        "has_missing_rating",
        F.col("rating").isNull()
    )
)

display(
    df_silver.select(
        "product_id",
        "product_name",
        "price_brl",
        "rating",
        "has_missing_price",
        "has_missing_rating"
    )
)

product_id,product_name,price_brl,rating,has_missing_price,has_missing_rating
KB001,AGE-R Booster Pro X2,1899.9,4.8,false,false
KB002,AGE-R PDRN Booster Gel,149.9,4.7,false,false
KB003,Acne Master Patch,39.9,4.6,false,false
KB004,Concentrated Ginseng Mask,89.9,null,false,true
KB005,Vitamin C Expert 25 Ampoule,null,4.9,true,false


In [0]:
df_silver.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_silver.columns
]).show()

+-----+--------+--------+---------------+---------+----------+------------+------+-----------------+------------------+
|brand|category|in_stock|key_ingredients|price_brl|product_id|product_name|rating|has_missing_price|has_missing_rating|
+-----+--------+--------+---------------+---------+----------+------------+------+-----------------+------------------+
|    0|       0|       0|              0|        1|         0|           0|     1|                0|                 0|
+-----+--------+--------+---------------+---------+----------+------------+------+-----------------+------------------+



In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("kbeauty_silver_products")

In [0]:
spark.sql("""
SELECT *
FROM kbeauty_silver_products
""").show(truncate=False)

+---------+-------------+--------+----------------+---------+----------+---------------------------+------+-----------------+------------------+
|brand    |category     |in_stock|key_ingredients |price_brl|product_id|product_name               |rating|has_missing_price|has_missing_rating|
+---------+-------------+--------+----------------+---------+----------+---------------------------+------+-----------------+------------------+
|Sulwhasoo|Skincare     |true    |[Ginseng]       |89.9     |KB004     |Concentrated Ginseng Mask  |NULL  |false            |true              |
|Medicube |Skincare     |true    |[PDRN]          |149.9    |KB002     |AGE-R PDRN Booster Gel     |4.7   |false            |false             |
|Iope     |Skincare     |true    |[Vitamin C]     |NULL     |KB005     |Vitamin C Expert 25 Ampoule|4.9   |true             |false             |
|Medicube |Beauty Device|true    |[PDRN, Peptides]|1899.9   |KB001     |AGE-R Booster Pro X2       |4.8   |false            |false